# GUNDAM interface demo

This notebook configures GUNDAM through `gundam-interface`, initializes the fitter engine, evaluates the nominal likelihood, runs a minimization, and inspects the best-fit parameters.


In [ ]:
nCpuThreads = 3
gundamLibPath = "/Users/nadrino/Documents/Work/Install/gundam/lib"
workDir = "/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024"
configPath = "configOA2024.yaml"
overrideList = [
    "override/v12ProdRun45.yaml",
    "override/onlyFlux5.yaml",
    "override/noEigen.yaml",
]
dataType = "Asimov"  # "Asimov", "Toy", or "RealData"
seed = 12345

In [ ]:
import sys
import numpy as np
from pathlib import Path

# Prefer the local checkout when running this notebook before pip installation.
# If you install this package with pip, you can remove this block.
# User configuration
repoRoot = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
srcPath = repoRoot / "src"
if srcPath.exists() and str(srcPath) not in sys.path:
    sys.path.insert(0, str(srcPath))
# ~ end of this block

from gundam_interface import GundamInterface, GundamLoader, GundamRuntime


In [ ]:
np.random.seed(seed)

runtime = GundamRuntime(
    loader=GundamLoader(gundamLibPath=gundamLibPath),
    workDir=workDir,
    nCpuThreads=nCpuThreads,
    configPath=configPath,
    overrideList=overrideList,
    dataType=dataType,
    randomSeed=seed,
)

runtime.toDict(includeConfigJsonString=False)


In [ ]:
gundam = GundamInterface(runtime)
gundam.configure()
gundam.initialize()

print(f"Initialized GUNDAM with {len(gundam.parameters)} active parameters")
for index, parameter in enumerate(gundam.parameters):
    print(
        f"{index:3d}: {parameter.name} "
        f"prior={parameter.prior:.8g} "
        f"step={parameter.stepSize:.8g} "
        f"value={parameter.value:.8g}"
    )


In [ ]:
nominalPhysicalValues = gundam.getParameterValues()
nominalLlh = gundam.evaluateLlh()

print(f"Nominal LLH: {nominalLlh:.8g}")
print("Nominal parameters:")
for name, physicalValue in zip(gundam.parameterNames, nominalPhysicalValues):
    print(f"  - {name}: physical={physicalValue:.8g}")


In [ ]:
bestFitLlh = gundam.minimize()
bestFitPhysicalValues = gundam.getParameterValues()
deltaLlh = bestFitLlh - nominalLlh

print(f"Best-fit LLH: {bestFitLlh:.8g}")
print(f"Delta LLH relative to nominal: {deltaLlh:.8g}")


In [ ]:
print("Best-fit parameters:")
for name, physicalValue in zip(gundam.parameterNames, bestFitPhysicalValues):
    print(f"  - {name}: physical={physicalValue:.8g}")


In [ ]:
# Re-evaluate the likelihood at the best-fit point as a consistency check.
reevaluatedBestFitLlh = gundam.evaluateLlh(physicalValues=bestFitPhysicalValues)

print(f"Minimizer best-fit LLH: {bestFitLlh:.8g}")
print(f"Re-evaluated best-fit LLH: {reevaluatedBestFitLlh:.8g}")
print(f"Absolute difference: {abs(reevaluatedBestFitLlh - bestFitLlh):.8g}")
